Markdown:

#***Audio Data Processing in Google Colab*** ~

This notebook demonstrates how to load and handle an underwater audio dataset.  

**Dataset Loading Methods:**

Direct Upload: Good for small, standalone .wav files.

 Google Drive Mounting (Recommended): Faster and preserves the dataset's complete folder structure without requiring constant re-uploads.  

** Audio Analysis with Librosa & Pathlib:**

 librosa: Used over soundfile for advanced machine learning analysis (e.g., spectrograms). Always set sr=None to preserve the original audio sampling rate.  

 pathlib.rglob(): Recursively iterates through and fetches all nested .wav files within the directories.  

In [ ]:
from google.colab import files

uploaded = files.upload()


This code snippet is used to fetch a specific .wav file from the local machine.


In [ ]:
import librosa

audio, sr = librosa.load("Bluewhale.wav", sr= None)

print(audio.shape)
print(sr)

**import librosa**:
imports the librosa library, which is used for audio and music processing.

**"How is it better than soundfile"**: soundfile is just used for reading and writing audio files, however librosa includes analysis of audio files, hence it is more widely used in machine learning.Creating Spectograms, Extracting MFCCs etc are included in librosa. Librosa uses soundfile internally.

**"sr= None"** has to be written because Librosa automatically changes the sampling rate(sr) of the audio files. Hence to keep it original, we mention sr=None, otherwise the sr always comes out to be 22050.

**"audio.shape"**: shape refers to the total number of samples in the audio file.

         (sampling rate * number of seconds) = shape


In [ ]:
from google.colab import files

uploaded = files.upload()

In order to fetch the entire dataset while maintaining the folder structure, there are two methods:
This is the first one **Converting the entire data folder inta a zip file**
This method is easier however very time consuming if the dataset is large, hence the second approach is recommended

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

This is the second method: **Using google drive** to fetch the entire dataset with the folder structure.

The dataset is mounted at the location content/drive/Mydrive.

This method is better as it is faster and we wouldnt have to reupload the zip file everytime.

If we want to access a certain file in the dataset, we can easily do so as we can see in the next cell->

In [ ]:
import librosa

audio, sr = librosa.load(
    "/content/drive/MyDrive/Underwater Audio Data/data/Biological/Fish_1.wav",
    sr=None
)
print(audio.shape)
print(sr)

In order to loop through and read all files in a particular folder, refer to the next cell->

In [ ]:
from pathlib import Path

dataset_path = Path("/content/drive/MyDrive/Underwater Audio Data/data/Biological")

for wav_file in dataset_path.rglob("*.wav"):
    print(wav_file)

In the above cell we use the pathlib library path class that helps work with folders and files.

**("*.wav")** it helps finding all the files ending with .wav

**rglob**: There are two types of functions: glob() and rglob().
glob() means global, it searches the file inside the given folder
rglob() mean recursive global, which searches recursively inside folders presnt in the given folder and even the folders inside them and so on.

Now lets calculate the total number of audio files present in each class of the Raw dataset.


In [ ]:
from pathlib import Path

# Define the base path to your raw dataset classes
base_dataset_path = Path("/content/drive/MyDrive/Underwater Audio Data/data/")

# Dictionary to store the count of files for each class
class_file_counts = {}

# Iterate through each subdirectory (class) in the base dataset path
for class_folder in base_dataset_path.iterdir():
    if class_folder.is_dir(): # Ensure it's a directory
        class_name = class_folder.name
        # Count all .wav files recursively within the class folder
        wav_files_in_class = list(class_folder.rglob("*.wav"))
        class_file_counts[class_name] = len(wav_files_in_class)

# Display the results
print("Number of audio files per class:")
for class_name, count in class_file_counts.items():
    print(f"- {class_name}: {count} files")


Now lets Calculate the total duration of audio in each class.

In [ ]:
import librosa
from pathlib import Path

base_dataset_path = Path("/content/drive/MyDrive/Underwater Audio Data/data/")

# Add a check for the path before proceeding
if not base_dataset_path.exists():
    print(f"Error: The directory '{base_dataset_path}' does not exist. Please ensure your Google Drive is mounted and the path is correct.")
    raise FileNotFoundError(f"Directory not found: {base_dataset_path}")
if not base_dataset_path.is_dir():
    print(f"Error: The path '{base_dataset_path}' is not a directory. Please verify the path.")
    raise NotADirectoryError(f"Path is not a directory: {base_dataset_path}")

# Dictionary to store the total duration for each class
class_total_durations = {}

# Iterate through each subdirectory (class) in the base dataset path
for class_folder in base_dataset_path.iterdir():
    if class_folder.is_dir():  # Ensure it's a directory
        class_name = class_folder.name
        total_duration_seconds = 0

        # Iterate through all .wav files recursively within the class folder
        for wav_file in class_folder.rglob("*.wav"):
            try:
                # Get duration without loading the full audio data
                duration = librosa.get_duration(path=str(wav_file))
                total_duration_seconds += duration
            except Exception as e:
                print(f"Error processing {wav_file}: {e}")
                continue

        class_total_durations[class_name] = total_duration_seconds

# Display the results
print("Total audio duration per class:")
for class_name, duration in class_total_durations.items():
    # Convert duration from seconds to minutes for better readability
    duration_minutes = duration / 60
    print(f"- {class_name}: {duration_minutes:.2f} minutes ({duration:.2f} seconds)")
